# Atividade: limpeza e organização de dados de vendas

Objetivo: transformar `vendas_nao_tratadas.csv` em uma base consistente. Use separador `;` e codificação UTF-8.

## 1. Importação e diagnóstico

Importe pandas, carregue o CSV e examine dimensões, tipos, primeiras linhas, valores ausentes e duplicatas.

In [ ]:
import pandas as pd

df = pd.read_csv('vendas_nao_tratadas.csv', sep=';', encoding='utf-8-sig', dtype=str)

print("--- DIAGNÓSTICO INICIAL ---")
print(f"Dimensões originais: {df.shape[0]} linhas, {df.shape[1]} colunas\n")
print(f"Total de duplicatas exatas: {df.duplicated().sum()}")
print("\nValores nulos por coluna:\n", df.isna().sum())

--- DIAGNÓSTICO INICIAL ---
Dimensões originais: 300 linhas, 12 colunas

Total de duplicatas exatas: 8

Valores nulos por coluna:
 id_venda            0
data_venda          0
cliente             0
email               0
cidade              0
uf                  0
produto             0
categoria           0
quantidade         12
preco_unitario      0
forma_pagamento     0
status_pedido       0
dtype: int64


## 2. Padronização textual

Remova espaços extras; padronize cliente em formato de nome, cidade/categoria com capitalização adequada, UF em maiúsculas, forma de pagamento e status.

In [ ]:
for col in df.columns:
    df[col] = df[col].str.strip()

df['cliente'] = df['cliente'].str.title()
df['cidade'] = df['cidade'].str.title()
df['categoria'] = df['categoria'].str.title()
df['uf'] = df['uf'].str.upper()

df['forma_pagamento'] = df['forma_pagamento'].str.capitalize()
df['status_pedido'] = df['status_pedido'].str.capitalize()

mapeamento_pagamento = {
    'Cartao debito': 'Cartão de Débito',
    'Cartao credito': 'Cartão de Crédito',
    'Cartão de crédito': 'Cartão de Crédito',
    'Cartão de débito': 'Cartão de Débito',
    'Boleto': 'Boleto',
    'Pix': 'Pix'
}

mapeamento_status = {
    'Concluido': 'Concluído',
    'Concluído': 'Concluído',
    'Pendente': 'Pendente',
    'Cancelado': 'Cancelado'
}

df['forma_pagamento'] = df['forma_pagamento'].replace(mapeamento_pagamento)
df['status_pedido'] = df['status_pedido'].replace(mapeamento_status)

print("--- VISUALIZAÇÃO APÓS PADRONIZAÇÃO TEXTUAL ---")
print(df[['cliente', 'cidade', 'uf', 'categoria', 'forma_pagamento', 'status_pedido']].head(), "\n")

--- VISUALIZAÇÃO APÓS PADRONIZAÇÃO TEXTUAL ---
            cliente          cidade  uf    categoria    forma_pagamento  \
0  Henrique Martins        Curitiba  PR       Móveis  Cartão de Crédito   
1    Patrícia Nunes       Fortaleza  CE   Acessórios             Boleto   
2        Bruno Lima        Campinas  SP  Eletrônicos   Cartão de Débito   
3   Isabela Ribeiro  Belo Horizonte  MG       Móveis                Pix   
4   Rafael Teixeira          Recife  PE   Acessórios  Cartão de Crédito   

  status_pedido  
0     Concluído  
1     Concluído  
2      Pendente  
3     Cancelado  
4     Concluído   



## 3. Conversão de tipos

Converta datas aceitando `AAAA-MM-DD` e `DD/MM/AAAA`; troque vírgula decimal por ponto; converta quantidade e preço para números.


In [ ]:
# Datas
df['data_venda'] = pd.to_datetime(df['data_venda'], format='mixed', dayfirst=True, errors='coerce')

def limpar_preco(valor):
    if pd.isna(valor): return valor
    valor_str = str(valor)
    if ',' in valor_str:
        valor_str = valor_str.replace('.', '')
        valor_str = valor_str.replace(',', '.')
    return valor_str

df['preco_unitario'] = df['preco_unitario'].str.replace('R$', '', regex=False).str.strip()
df['preco_unitario'] = df['preco_unitario'].apply(limpar_preco)
df['preco_unitario'] = pd.to_numeric(df['preco_unitario'], errors='coerce')

df['quantidade'] = pd.to_numeric(df['quantidade'], errors='coerce')

print("--- VISUALIZAÇÃO APÓS CONVERSÃO DE TIPOS ---")
print("Tipos de dados atualizados:")
print(df[['data_venda', 'quantidade', 'preco_unitario']].dtypes, "\n")
print("Primeiras linhas:")
print(df[['data_venda', 'quantidade', 'preco_unitario']].head(), "\n")

--- VISUALIZAÇÃO APÓS CONVERSÃO DE TIPOS ---
Tipos de dados atualizados:
data_venda        datetime64[us]
quantidade               float64
preco_unitario           float64
dtype: object 

Primeiras linhas:
  data_venda  quantidade  preco_unitario
0 2025-01-08         NaN           899.9
1 2025-01-15         3.0           389.5
2 2025-01-22         4.0           319.0
3 2025-01-29         5.0          1490.0
4 2025-05-02         1.0           129.9 



## 4. Validação

Identifique e-mails inválidos, quantidades/preços ausentes ou não positivos e datas inválidas. Decida como tratar e documente.

In [ ]:
print("--- RELATÓRIO ANTES DO TRATAMENTO ---")
print(f"Total de linhas atual: {df.shape[0]}")

padrao_email = r'^[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+$'
mascara_email_invalido = ~df['email'].str.match(padrao_email, na=False)

mascara_qtd_invalida = df['quantidade'].isna() | (df['quantidade'] <= 0)
mascara_preco_invalido = df['preco_unitario'].isna() | (df['preco_unitario'] <= 0)
mascara_data_invalida = df['data_venda'].isna()

print(f"E-mails inválidos: {mascara_email_invalido.sum()}")
print(f"Quantidades inválidas/ausentes: {mascara_qtd_invalida.sum()}")
print(f"Preços inválidos/ausentes: {mascara_preco_invalido.sum()}")
print(f"Datas inválidas: {mascara_data_invalida.sum()}\n")

df = df[~mascara_email_invalido & ~mascara_qtd_invalida & ~mascara_preco_invalido & ~mascara_data_invalida]

print("--- RELATÓRIO APÓS O TRATAMENTO ---")
print(f"Linhas restantes (ainda com duplicatas): {df.shape[0]}\n")

--- RELATÓRIO ANTES DO TRATAMENTO ---
Total de linhas atual: 300
E-mails inválidos: 10
Quantidades inválidas/ausentes: 12
Preços inválidos/ausentes: 0
Datas inválidas: 0

--- RELATÓRIO APÓS O TRATAMENTO ---
Linhas restantes (ainda com duplicatas): 279



## 5. Duplicatas e organização

Remova duplicatas exatas, ordene por data e id, redefina o índice e deixe as colunas na ordem original.

In [ ]:
print("--- RELATÓRIO ANTES DA ORGANIZAÇÃO ---")
print(f"Total de linhas atual: {df.shape[0]}")
print(f"Duplicatas exatas encontradas: {df.duplicated().sum()}\n")

df = df.drop_duplicates()

df = df.sort_values(by=['data_venda', 'id_venda'])

colunas_originais = [
    'id_venda', 'data_venda', 'cliente', 'email', 'cidade', 'uf', 
    'produto', 'categoria', 'quantidade', 'preco_unitario', 
    'forma_pagamento', 'status_pedido'
]
df = df[colunas_originais]
df = df.reset_index(drop=True)

print("--- RELATÓRIO APÓS A ORGANIZAÇÃO ---")
print(f"Linhas restantes (sem duplicatas): {df.shape[0]}\n")

--- RELATÓRIO ANTES DA ORGANIZAÇÃO ---
Total de linhas atual: 279
Duplicatas exatas encontradas: 7

--- RELATÓRIO APÓS A ORGANIZAÇÃO ---
Linhas restantes (sem duplicatas): 272



## 6. Verificação e exportação

Faça testes de qualidade e exporte o resultado com separador `;`, UTF-8 com BOM e sem índice.

In [84]:
print("--- TESTES DE QUALIDADE FINAIS ---")
print(f"Total de registros: {df.shape[0]}")
print(f"Total de colunas: {df.shape[1]}")
print(f"Valores nulos:\n{df.isna().sum()}")

df.to_csv('vendas_tratadas.csv', sep=';', encoding='utf-8-sig', index=False)
print("\nArquivo exportado com sucesso")

--- TESTES DE QUALIDADE FINAIS ---
Total de registros: 272
Total de colunas: 12
Valores nulos:
id_venda           0
data_venda         0
cliente            0
email              0
cidade             0
uf                 0
produto            0
categoria          0
quantidade         0
preco_unitario     0
forma_pagamento    0
status_pedido      0
dtype: int64

Arquivo exportado com sucesso
